# TabDPT Regressor — Artifact Inference Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-regressor-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/tutorials/tabdpt_regressor_artifact_inference_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Layer6%2FTabDPT-ffcc4d?style=flat)](https://huggingface.co/Layer6/TabDPT)
[![Upstream](https://img.shields.io/badge/Upstream-layer6ai--labs%2FTabDPT--inference-181717?style=flat&logo=github&logoColor=white)](https://github.com/layer6ai-labs/TabDPT-inference)
[![arXiv](https://img.shields.io/badge/arXiv-2608.01400-b31b1b.svg)](https://arxiv.org/abs/2608.01400)

This tutorial demonstrates how to load an exported DIMER serving artifact bundle (`artifact.json` + `training_context.csv`) in a fresh process, condition the in-context TabDPT foundation model on the saved support table, and run batch inference on new unlabelled regression records without refitting.

In [ ]:
!git clone -q https://github.com/kurtvalcorza/tabdpt-regressor-pipeline.git /content/tabdpt-regressor-pipeline
%pip install -q '/content/tabdpt-regressor-pipeline[model]'


## 1. Prepare and inspect the DIMER serving artifact bundle

In DIMER, a completed training/export run produces an `artifact.json` manifest referencing the support context table `training_context.csv` and the pinned base model metadata.

In [ ]:
import hashlib
import json
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from tabdpt_regressor_pipeline import TabDPTRegressionPipeline

# Prepare a sample dataset split for demonstration
frame = load_diabetes(as_frame=True).frame
train, test = train_test_split(frame, test_size=0.2, random_state=42)

bundle_dir = Path('/content/artifacts')
bundle_dir.mkdir(parents=True, exist_ok=True)
context_path = bundle_dir / 'training_context.csv'
train.to_csv(context_path, index=False)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    'format': 'tabdpt-dimer-context-v2',
    'taskType': 'tabular_regression',
    'targetColumn': 'target',
    'dropColumns': [],
    'baseModel': {
        'repo': 'Layer6/TabDPT',
        'revision': '4462ffbd1d8dea25d4862d30beed4b70cd596ae5',
        'filename': 'tabdpt1_2.safetensors',
        'sha256': '06680220fd66c4524051706b98c1c659a674d19d3a766cd0bb276505e99faccd'
    },
    'trainingContext': {
        'path': 'training_context.csv',
        'sha256': sha256_file(context_path)
    }
}
manifest_path = bundle_dir / 'artifact.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print('Serving artifact bundle initialized at:', bundle_dir)


## 2. Load artifact bundle and condition the model

We verify the cryptographic digest of `training_context.csv` against `artifact.json`, construct `TabDPTRegressionPipeline` with explicit `use_flash=False` for GPU portability, and condition it on the support context.

In [ ]:
loaded_manifest = json.loads(manifest_path.read_text())
observed_context_sha = sha256_file(context_path)
assert observed_context_sha == loaded_manifest['trainingContext']['sha256'], 'Context digest mismatch!'

context_df = pd.read_csv(context_path)
pipe = TabDPTRegressionPipeline(compile_model=False, use_flash=False)
pipe.fit(
    context_df,
    target_column=loaded_manifest['targetColumn'],
    drop_columns=loaded_manifest.get('dropColumns', [])
)
print('Model ready. Target column:', pipe.target_column)


## 3. Score unlabelled observations

We drop the target column from the test set to simulate real-world unlabelled batch scoring, obtaining predicted continuous target values.

In [ ]:
unlabelled_test = test.drop(columns=[loaded_manifest['targetColumn']])
predictions = pipe.predict(unlabelled_test, n_ensembles=2, context_size=512, batch_size=512, seed=42)

print('Predictions count:', len(predictions))
print('Sample predictions (first 5):')
print(predictions.head(5))


In [ ]:
# Evaluate against ground-truth continuous targets
metrics = pipe.evaluate(test, n_ensembles=2, context_size=512, batch_size=512, seed=42)
print('Holdout evaluation metrics:', metrics)


## Production notes

For production deployment in DIMER:
1. Because TabDPT is an in-context learner, the served model consists of `tabdpt1_2.safetensors` + `training_context.csv` + `artifact.json`.
2. Treat `training_context.csv` with the same governance and security controls as the original training dataset.
3. Setting `use_flash=False` ensures portability across both Tesla T4 (`sm_75`) GPUs and newer Ampere/Hopper (`sm_80+`) architectures.
